# Benchmark GCN on GHZ_a and GHZ_b


### Introduction

This notebook walks through reproducing the key results from the paper.

## 1. Import Dependencies

First, we import the necessary dependencies for our evaluation.

In [1]:
# Suppress PyTorch Warnings and ensure proper dependency loading
import warnings
import os
import sys

warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=UserWarning)
project_root = os.path.abspath(os.path.join(os.getcwd(), '../../..')) # Go two levels up from the notebook location to reach project root

if project_root not in sys.path:
    sys.path.insert(0, project_root)
print("Project root set to:", project_root)

Project root set to: C:\SQuASH


In [2]:
import os
import torch
import json
from torch_geometric.data import DataLoader

from config import DeviceConfig, PathConfig, get_model_config_from_path
from surrogate_models.architectures.gnn.gcn_runner import prepare_paths_and_config, set_seed
from evaluate.evaluate_utils import compute_metrics, evaluate_gnn, load_gnn_model

In [3]:
# Initialize device configuration and path configuration
device_config = DeviceConfig()
device        = device_config.device
path_config   = PathConfig()

## 2. Data Loading Utils

Let´s define function to load corresponding test data from [Zenodo](https://zenodo.org/records/15551805).  
> ⚠️ **Note: If you've already downloaded the data before, please skip this section and data loading cells in the evaluation.**

In [35]:
!pip install requests

In [4]:
import requests
import zipfile
import io
import os

In [5]:
def zenodo_load_data():
    dataset_names = [
    f"test_{search_space}_proxy_squash",
    f"test_{search_space}_squash"
    ]
    for d in dataset_names:
        target_file_name = f'{d}.pt'
        target_path = os.path.join(target_directory, target_file_name)
        if os.path.isfile(target_path):
            print(f"File '{target_file_name}' already exists at '{target_directory}'. Skipping download.")
        else:
            zip_url = f"https://zenodo.org/record/15551805/files/graph_data_{search_space}.zip"
        
            print(f"Downloading ZIP `graph_data_{search_space}.zip` from https://zenodo.org/ ...")
            response = requests.get(zip_url)
            response.raise_for_status()  # Raise an error if the download failed
        
            with zipfile.ZipFile(io.BytesIO(response.content)) as z:
                print("Contents of the ZIP file:")
                for file_name in z.namelist():
                    print(f" - {file_name}")
        
                if target_file_name in z.namelist():
                    print(f"Extracting {target_file_name}...")
                    os.makedirs(target_directory, exist_ok=True)
                    z.extract(target_file_name, path=target_directory)
                    print(f"Saved '{target_file_name}' to '{target_directory}'.")
                else:
                    print(f"File '{target_file_name}' not found in the archive.")

In [6]:
target_directory = os.path.abspath('../../../data/processed_data/gcn_processed_data') 

## 3. Metrics Calculation


To start, we define the function that computes the necessary metrics.

In [10]:
def reproduce_results(search_space):
    dataset_names = [
    f"test_{search_space}_proxy_squash",
    f"test_{search_space}_squash",
    f"test_{search_space}_proxy_squash",
    f"test_{search_space}_squash",
    ]
    model_names = [
    f"gcn_proxy_{search_space}",
    f"gcn_{search_space}",
    f"gcn_proxy_augmented_{search_space}",
    f"gcn_augmented_{search_space}",
    ]
    # Loop over each dataset/model pair
    for data_set, model_name in zip(dataset_names, model_names):
       print(f"\n=== Dataset: {data_set} with Model: {model_name} ===")     
       # load the data     
       data_path = os.path.join(
                path_config.paths['gcn_data'],
                f"{data_set}.pt"
            )    
       circuits   = torch.load(data_path, weights_only=False)           # list of Data objects
       loader     = DataLoader(circuits, batch_size=32, shuffle=False)
        
       # load model config
       cfg_path   = os.path.join(
                os.path.join(path_config.paths['benchmark_search_spaces'], f'{search_space}/surrogate_models/configs', f'{model_name}_config.json')
       )
       model_path = os.path.join(path_config.paths['benchmark_search_spaces'], f'{search_space}/surrogate_models', f"{model_name}.pth")
       model_cfg  = get_model_config_from_path(cfg_path, device)
        
       # instantiate RegGNN
       model = load_gnn_model(model_path, model_cfg)
       model.to(device).eval()
        
       # inference
       preds, labels  = evaluate_gnn(circuits, model, model_cfg)
       
       # compute and print metrics
       compute_metrics(preds, labels, label=model_name, tolerance=0.1)

## 4. Reproduce Evaluation for Search Space `ghz_a`

First, we load test data for `ghz_a` with the functions defined in Sec.2.

In [11]:
search_space = "ghz_a"

In [6]:
zenodo_load_data()

Contents of the ZIP file:
 - test_ghz_a_proxy_squash.pt
 - test_ghz_a_squash.pt
 - train_ghz_a_augmented_proxy_squash.pt
 - train_ghz_a_augmented_squash.pt
 - train_ghz_a_proxy_squash.pt
 - train_ghz_a_squash.pt
 - val_ghz_a_augmented_proxy_squash.pt
 - val_ghz_a_augmented_squash.pt
 - val_ghz_a_proxy_squash.pt
 - val_ghz_a_squash.pt
Extracting test_ghz_a_proxy_squash.pt...
Saved 'test_ghz_a_proxy_squash.pt' to 'C:\SQuASH\data\processed_data\gcn_processed_data'.
Contents of the ZIP file:
 - test_ghz_a_proxy_squash.pt
 - test_ghz_a_squash.pt
 - train_ghz_a_augmented_proxy_squash.pt
 - train_ghz_a_augmented_squash.pt
 - train_ghz_a_proxy_squash.pt
 - train_ghz_a_squash.pt
 - val_ghz_a_augmented_proxy_squash.pt
 - val_ghz_a_augmented_squash.pt
 - val_ghz_a_proxy_squash.pt
 - val_ghz_a_squash.pt
Extracting test_ghz_a_squash.pt...
Saved 'test_ghz_a_squash.pt' to 'C:\SQuASH\data\processed_data\gcn_processed_data'.


Next, we define paths and other configurations necessary for loading the model and dataset, and set the random seed.

In [12]:
# also load the full JSON + timestamp if you like
config, gate_set_name, timestamp = prepare_paths_and_config(search_space, device)
print(f"\n=== Search‐space: {search_space} ({timestamp}) ===")
print("Config:")
print(json.dumps(config, indent=2, default=str))


=== Search‐space: ghz_a (2025-06-04_18-27-09) ===
Config:
{
  "device": "cpu",
  "seed": 42,
  "runseed": 42,
  "batch_size": 32,
  "num_workers": 0,
  "epochs": 3,
  "emb_dim": 1200,
  "layer_num": 8,
  "qubit_num": 3,
  "num_node_features": 7,
  "drop_ratio": 0.012714767230404513,
  "lr": 0.00042048670814195114,
  "decay": 1.2239395743425164e-06,
  "JK": "mean",
  "patience": 7,
  "metric": "spearman",
  "graph_pooling": "max",
  "n_estimators": null,
  "max_depth": null,
  "random_state": null,
  "optuna_trials": null,
  "min_samples_split": null,
  "min_samples_leaf": null,
  "max_features": null,
  "n_jobs": null,
  "PATHS": {
    "optuna_studies": "C:\\SQuASH\\surrogate_models/tuning\\studies",
    "raw_data": "C:\\SQuASH\\data/raw_data/",
    "gcn_data": "C:\\SQuASH\\data/processed_data/gcn_processed_data",
    "rf_data": "C:\\SQuASH\\data/processed_data/rf_processed_data",
    "trained_models": "C:\\SQuASH\\surrogate_models\\trained_models",
    "benchmark_search_spaces": "C:\

In [13]:
set_seed(config["runseed"])

Finally, we run the evaluation for the models presented in the paper, i.e.,  `GCN` -> gcn_ghz_a, `GCN_pr` -> gcn_proxy_ghz_a, `GCN_aug`, `GCN_pr_aug`.

In [14]:
reproduce_results(search_space=search_space)


=== Dataset: test_ghz_a_proxy_squash with Model: gcn_proxy_ghz_a ===
--- Metrics for gcn_proxy_ghz_a ---
Samples: 82926
MSE:     0.0020
MAE:     0.0193
RMSE:    0.0449
R^2:     0.9389
Corr:    0.9694
Spearman: 0.8314
Accuracy (|err| <= 0.1): 96.01%


=== Dataset: test_ghz_a_squash with Model: gcn_ghz_a ===
--- Metrics for gcn_ghz_a ---
Samples: 82926
MSE:     0.0025
MAE:     0.0296
RMSE:    0.0499
R^2:     0.9246
Corr:    0.9681
Spearman: 0.8320
Accuracy (|err| <= 0.1): 95.84%


=== Dataset: test_ghz_a_proxy_squash with Model: gcn_proxy_augmented_ghz_a ===
--- Metrics for gcn_proxy_augmented_ghz_a ---
Samples: 82926
MSE:     0.0023
MAE:     0.0269
RMSE:    0.0480
R^2:     0.9303
Corr:    0.9693
Spearman: 0.8318
Accuracy (|err| <= 0.1): 95.95%


=== Dataset: test_ghz_a_squash with Model: gcn_augmented_ghz_a ===
--- Metrics for gcn_augmented_ghz_a ---
Samples: 82926
MSE:     0.0018
MAE:     0.0183
RMSE:    0.0425
R^2:     0.9453
Corr:    0.9724
Spearman: 0.8318
Accuracy (|err| <= 0.1): 

## 4. Reproduce Evaluation for Search Space `ghz_b`

In [15]:
search_space = "ghz_b"

In [7]:
# Load test data for ghz_b
zenodo_load_data()

Contents of the ZIP file:
 - test_ghz_b_proxy_squash.pt
 - test_ghz_b_squash.pt
 - train_ghz_b_aug_proxy_squash.pt
 - train_ghz_b_aug_squash.pt
 - train_ghz_b_proxy_squash.pt
 - train_ghz_b_squash.pt
 - val_ghz_b_aug_proxy_squash.pt
 - val_ghz_b_aug_squash.pt
 - val_ghz_b_proxy_squash.pt
 - val_ghz_b_squash.pt
Extracting test_ghz_b_proxy_squash.pt...
Saved 'test_ghz_b_proxy_squash.pt' to 'C:\SQuASH\data\processed_data\gcn_processed_data'.
Contents of the ZIP file:
 - test_ghz_b_proxy_squash.pt
 - test_ghz_b_squash.pt
 - train_ghz_b_aug_proxy_squash.pt
 - train_ghz_b_aug_squash.pt
 - train_ghz_b_proxy_squash.pt
 - train_ghz_b_squash.pt
 - val_ghz_b_aug_proxy_squash.pt
 - val_ghz_b_aug_squash.pt
 - val_ghz_b_proxy_squash.pt
 - val_ghz_b_squash.pt
Extracting test_ghz_b_squash.pt...
Saved 'test_ghz_b_squash.pt' to 'C:\SQuASH\data\processed_data\gcn_processed_data'.


Again, we begin by defining our search space, paths and other configurations necessary for loading the model and dataset.

In [16]:
# Load the full JSON + timestamp if you like
config, gate_set_name, timestamp = prepare_paths_and_config(search_space, device)
set_seed(config["runseed"])

print(f"\n=== Search‐space: {search_space} ({timestamp}) ===")
print("Config:")
print(json.dumps(config, indent=2, default=str))


=== Search‐space: ghz_b (2025-06-04_19-39-54) ===
Config:
{
  "device": "cpu",
  "seed": 42,
  "runseed": 42,
  "batch_size": 128,
  "num_workers": 0,
  "epochs": 100,
  "emb_dim": 1050,
  "layer_num": 8,
  "qubit_num": 3,
  "num_node_features": 8,
  "drop_ratio": 0.0644893118913786,
  "lr": 4.540520885756229e-05,
  "decay": 1.917208797826118e-06,
  "JK": "mean",
  "patience": 7,
  "metric": "spearman",
  "graph_pooling": "attention",
  "n_estimators": null,
  "max_depth": null,
  "random_state": null,
  "optuna_trials": null,
  "min_samples_split": null,
  "min_samples_leaf": null,
  "max_features": null,
  "n_jobs": null,
  "PATHS": {
    "optuna_studies": "C:\\SQuASH\\surrogate_models/tuning\\studies",
    "raw_data": "C:\\SQuASH\\data/raw_data/",
    "gcn_data": "C:\\SQuASH\\data/processed_data/gcn_processed_data",
    "rf_data": "C:\\SQuASH\\data/processed_data/rf_processed_data",
    "trained_models": "C:\\SQuASH\\surrogate_models\\trained_models",
    "benchmark_search_spaces":

In [17]:
reproduce_results(search_space=search_space)


=== Dataset: test_ghz_b_proxy_squash with Model: gcn_proxy_ghz_b ===
--- Metrics for gcn_proxy_ghz_b ---
Samples: 97753
MSE:     0.0074
MAE:     0.0491
RMSE:    0.0863
R^2:     0.8292
Corr:    0.9109
Spearman: 0.8717
Accuracy (|err| <= 0.1): 84.74%


=== Dataset: test_ghz_b_squash with Model: gcn_ghz_b ===
--- Metrics for gcn_ghz_b ---
Samples: 97753
MSE:     0.0072
MAE:     0.0467
RMSE:    0.0847
R^2:     0.8352
Corr:    0.9140
Spearman: 0.8743
Accuracy (|err| <= 0.1): 85.66%


=== Dataset: test_ghz_b_proxy_squash with Model: gcn_proxy_augmented_ghz_b ===
--- Metrics for gcn_proxy_augmented_ghz_b ---
Samples: 97753
MSE:     0.0069
MAE:     0.0457
RMSE:    0.0831
R^2:     0.8415
Corr:    0.9178
Spearman: 0.8776
Accuracy (|err| <= 0.1): 86.35%


=== Dataset: test_ghz_b_squash with Model: gcn_augmented_ghz_b ===
--- Metrics for gcn_augmented_ghz_b ---
Samples: 97753
MSE:     0.0069
MAE:     0.0451
RMSE:    0.0831
R^2:     0.8413
Corr:    0.9175
Spearman: 0.8758
Accuracy (|err| <= 0.1): 